# 17 — Dataset Assembly

Merges all training sources into one clean, balanced dataset ready for fine-tuning.

**Why merge?** Notebooks 03–09 accumulate high-quality, curated pairs in
`knowledge_pairs.jsonl`. Notebook 10 generates validated synthetic samples.
Without this merge step, the final fine-tune (notebook 12) would see only the
synthetic data — and all the curated pairs would go to waste.

**What this notebook does:**
1. Load validated synthetic samples from notebook 12 (`04_validated/valid_samples.jsonl`)
2. Load curated knowledge pairs from notebooks 03, 05, 06, 07, 08, 09, 11, and 12 (`02_knowledge/knowledge_pairs.jsonl`)
3. Convert everything to ChatML format with the canonical system prompt
4. Deduplicate by instruction prefix
5. Cap each task type using soft type-weighted caps (minority categories uncapped)
6. Shuffle and split into train / valid / test (80 / 10 / 10)
7. Write `mlx/` format ready for `mlx_lm lora --use-chat-template`

**Inputs:**
- `../data/04_validated/valid_samples.jsonl` — validated synthetic samples (from notebook 12)
- `../data/02_knowledge/knowledge_pairs.jsonl` — curated pairs (from notebooks 03, 05–09)

**Output:**
- `../data/05_dataset/train.jsonl` — training samples (full format with metadata)
- `../data/05_dataset/valid.jsonl` — validation samples
- `../data/05_dataset/test.jsonl`  — test samples
- `../data/05_dataset/mlx/train.jsonl` — messages-only ChatML, for `mlx_lm lora`
- `../data/05_dataset/mlx/valid.jsonl`
- `../data/05_dataset/stats.json`   — counts, task distribution, avg message lengths
- `run/<date>/16_dataset_assembly.csv` — per-sample metadata for analysis

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

import sys, importlib
_cfg_dir = SCRIPT_DIR
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

DATA_IN  = DATA_ROOT / '04_validated'
DATA_OUT = DATA_ROOT / '05_dataset'
MLX_DIR  = DATA_OUT / 'mlx'
DATA_OUT.mkdir(parents=True, exist_ok=True)
MLX_DIR.mkdir(parents=True, exist_ok=True)

# ── Load validated synthetic samples (notebook 09 output) ────────────────────
samples = []
valid_samples_file = DATA_IN / 'valid_samples.jsonl'
if not valid_samples_file.exists():
    raise FileNotFoundError(
        f'{valid_samples_file} not found — run notebook 12 first.'
    )
with open(valid_samples_file) as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f'Loaded {len(samples)} validated synthetic samples')
print('Distribution:', dict(Counter(s['task_type'] for s in samples)))

In [ ]:
# ── Load curated knowledge pairs (notebooks 03+05+06+07+08 output) ──────────
# These pairs are already high-quality (exec-validated or doc-grounded).
# They are not in the synthetic samples file, so they would be dropped without
# this merge step — meaning all the effort of NB06/06/07/08 would be wasted.

def _source_to_task_type(source):
    """Map knowledge_pairs source tags to task_type used in this notebook."""
    src = source.lower()
    if any(src.startswith(p) for p in ('book_qa:', 'wiki:', 'actions_explain', 'actions_which')):
        return 'syntax_qa'
    if src.startswith('repair'):
        return 'debugging'
    # example:*, mutation, recombination, spec_to_code, readme_to_code,
    # actions_usage, actions_alias, actions_context, book, aro_by_example, proposal:*
    return 'code_generation'


kp_added = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            p = json.loads(line)
            instruction = p.get('instruction', '').strip()
            output      = p.get('output', '').strip()
            if not instruction or not output:
                continue
            source    = p.get('source', 'knowledge_pairs')
            task_type = _source_to_task_type(source)
            samples.append({
                'task_type':  task_type,
                'instruction': instruction,
                'output':      output,
                'source':      source,
                'score':       p.get('score', 1.0),
                'valid':       True,   # already curated
            })
            kp_added += 1
else:
    print(f'WARNING: {PAIRS_FILE} not found — run notebooks 03, 05, 06, 07, 08 first.')

print(f'Added {kp_added} curated knowledge pairs')
print(f'Total samples before dedup: {len(samples)}')
print('Full distribution:', dict(Counter(s['task_type'] for s in samples)))

## System prompt

In [ ]:
# Load knowledge base and build the canonical system prompt.
# Uses build_system_prompt(kb) from config so NB10 uses the same prompt
# as NB10 and the warm-start NB05 (consistent system token across all training).
with open(KB_FILE) as _kb_f:
    _kb = json.load(_kb_f)

ARO_SYSTEM_PROMPT = build_system_prompt(_kb)
print(f'System prompt: {len(ARO_SYSTEM_PROMPT)} chars  ({len(_kb["actions"])} actions referenced)')

## Convert to ChatML

In [ ]:
def build_messages(sample):
    task  = sample.get('task_type', '')
    instr = sample.get('instruction', '').strip()
    inp   = sample.get('input', '').strip()
    out   = sample.get('output', '').strip()

    # Tool calling: already in multi-turn messages format
    if task == 'tool_calling':
        msgs = sample.get('messages', [])
        if msgs:
            return msgs  # Return full messages list
        return None

    if task == 'fim':
        prefix = sample.get('prefix', '')
        suffix = sample.get('suffix', '')
        middle = sample.get('middle', '')
        if middle.strip():
            user = (
                f'Complete the missing ARO statement(s). Only output the missing line(s).\n\n'
                f'```aro\n{prefix}\n<FILL_HERE>\n{suffix}\n```'
            )
            return user, middle
        elif instr:
            user = instr
            return user, out
        return None

    if not out:
        return None

    if inp:
        user = f'{instr}\n\n```aro\n{inp}\n```'
    else:
        user = instr

    return (user, out) if user.strip() else None


def to_chatml(sample):
    result = build_messages(sample)
    if result is None:
        return None

    # Multi-turn (tool calling): result is a list of messages
    if isinstance(result, list):
        msgs = result
        # Ensure system prompt is present and up-to-date
        if msgs and msgs[0].get('role') == 'system':
            msgs[0]['content'] = ARO_SYSTEM_PROMPT
        else:
            msgs.insert(0, {'role': 'system', 'content': ARO_SYSTEM_PROMPT})
        return {
            'messages': msgs,
            'task_type': sample.get('task_type', ''),
            'source':    sample.get('source', ''),
        }

    # Single-turn (code gen, Q&A, etc.): result is (user, assistant) tuple
    user_content, assistant_content = result
    if len(user_content) < 10 or len(assistant_content) < 10:
        return None
    return {
        'messages': [
            {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',      'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ],
        'task_type': sample.get('task_type', ''),
        'source':    sample.get('source', sample.get('domain', sample.get('scenario', ''))),
    }


chatml = [to_chatml(s) for s in samples]
chatml = [c for c in chatml if c is not None]
print(f'Converted: {len(chatml)}  (dropped {len(samples) - len(chatml)} empty)')
drop_rate = (len(samples) - len(chatml)) / max(1, len(samples))
if drop_rate > 0.15:
    print(f'⚠️  HIGH DROP RATE: {drop_rate:.0%} of samples lost during ChatML conversion.')
    print('   Check build_messages() for task types with missing fields.')

## Deduplicate

In [ ]:
seen, deduped = set(), []
for s in chatml:
    key = s['messages'][1]['content'][:300]
    if key not in seen:
        seen.add(key)
        deduped.append(s)
print(f'After dedup: {len(deduped)}')

## Soft type-weighted caps (minority categories uncapped)

In [ ]:
# Type-weighted caps: balanced for a coding assistant that also answers questions
# and calls tools. Code generation is the primary task but Q&A and tool calling
# are essential for `aro ask` to work well.
#
# Caps were softened (2026-04-22) after pipeline review found that the original
# caps dropped 84% of data (7,580 → 1,202).  Minority categories are now
# uncapped so we keep every sample we have; majority categories are doubled.
TYPE_CAPS = {
    'code_generation':     1200,   # doubled — primary task
    'syntax_qa':           900,    # doubled — knowledge Q&A
    'code_explanation':    None,   # uncapped (minority)
    'fim':                 None,   # uncapped (minority)
    'code_transformation': None,   # uncapped (minority)
    'tool_calling':        None,   # uncapped — critical for aro ask
    'debugging':           None,   # uncapped — always useful
    'correction':          None,   # uncapped — teaches model what actions DON'T exist
    'full_application':    None,   # uncapped — plan → complete multi-file app
}
DEFAULT_CAP = None   # uncapped by default for any new task types

capped, type_counts = [], Counter()
for s in deduped:
    t = s.get('task_type', 'unknown')
    cap = TYPE_CAPS.get(t, DEFAULT_CAP)
    if cap is None or type_counts[t] < cap:
        capped.append(s)
        type_counts[t] += 1

total_capped = len(capped)
print(f'After type-weighted cap: {total_capped}')
for t, n in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = 100 * n / max(1, total_capped)
    flag = '  ⚠️  DOMINANT' if pct > 45 else ''
    print(f'  {t:25s}: {n:5d}  ({pct:.1f}%){flag}')

# Guard: raise if any single task type exceeds 50% of training data
for t, n in type_counts.items():
    share = n / max(1, total_capped)
    if share > 0.65:
        raise ValueError(
            f'Task type "{t}" makes up {share:.0%} of the dataset — training will be skewed. '
            f'Lower the cap for "{t}" or add more data for other task types.'
        )

In [ ]:
import re as _re

def _is_plausible(sample):
    msgs = sample.get('messages', [])
    task = sample.get('task_type', '')

    # Tool calling samples are multi-turn with pre-validated format — skip content checks
    if task == 'tool_calling':
        return (len(msgs) >= 3, 'too_few_messages') if len(msgs) < 3 else (True, '')

    user = msgs[1]['content'] if len(msgs) > 1 else sample.get('instruction', '')
    asst = msgs[2]['content'] if len(msgs) > 2 else sample.get('output', '')

    if len(user.strip()) < 20 or len(asst.strip()) < 15:
        return False, 'too_short'
    if _re.match(r'^[!?.,\s]{10,}$', asst.strip()):
        return False, 'garbage_output'

    u_tok = set(user.lower().split())
    a_tok = set(asst.lower().split())
    if a_tok and len(a_tok & u_tok) / len(a_tok) > 0.85:
        return False, 'copy_ratio'

    # Code tasks MUST contain code blocks
    if task in ('code_generation', 'debugging', 'fim'):
        if '```' not in asst:
            return False, 'missing_code_block'

    # Q&A: must have explanatory text (not just code)
    if task == 'syntax_qa':
        text_only = _re.sub(r'```.*?```', '', asst, flags=_re.DOTALL).strip()
        if len(text_only) < 15:
            return False, 'qa_no_explanation'

    return True, ''

plausible = []
rejected_plausibility = Counter()
for s in capped:
    ok, reason = _is_plausible(s)
    if ok:
        plausible.append(s)
    else:
        rejected_plausibility[reason] += 1

print(f'Plausibility filter: {len(capped)} → {len(plausible)} ({len(capped)-len(plausible)} rejected)')
for reason, count in rejected_plausibility.most_common():
    print(f'  {reason}: {count}')

final = plausible

## Train / valid / test split  (80 / 10 / 10)

In [ ]:
# Split changed from 90/5/5 to 80/10/10 (2026-04-22) — the old 5% test set
# was only 38 samples, too small to measure changes under +-15 ppt.
# Complete-program filter — drop fragment-only code/debug samples.
# Fragments are the main reason students fail to wrap ARO in `(name: activity) { }`.
final, _frag_dropped = filter_complete_program_samples(final)
print(f'Complete-program filter: dropped {_frag_dropped} fragment-only code samples '
      f'(kept {len(final)})')

random.seed(42)
random.shuffle(final)
n     = len(final)
t_end = int(n * 0.80)
v_end = int(n * 0.90)
train = final[:t_end]
valid = final[t_end:v_end]
test  = final[v_end:]
print(f'train={len(train)}  valid={len(valid)}  test={len(test)}')

## Save full format + mlx format

In [ ]:
from corpus_sanitize import sanitize_assistant_text, gate_check

def _sanitize_record(rec):
    """Strip URLs/HTML/tool-signatures from assistant content in a sample."""
    msgs = rec.get('messages', [])
    for m in msgs:
        if isinstance(m, dict) and m.get('role') == 'assistant' and isinstance(m.get('content'), str):
            m['content'] = sanitize_assistant_text(m['content'])
    if isinstance(rec.get('output'), str):
        rec['output'] = sanitize_assistant_text(rec['output'])
    return rec

# Sanitise every split before writing — last defense against polluted
# samples reaching mlx-lm. Round-3 shipped a model that emitted
# `![](https://via.placeholder.com/...)` and `read_file(path)` because
# the corpus carried both patterns from scraped READMEs and from the
# system prompt baked into every training pair.
train = [_sanitize_record(s) for s in train]
valid = [_sanitize_record(s) for s in valid]
test  = [_sanitize_record(s) for s in test]

# Cleanliness gate — fail loudly if contamination still exceeds thresholds.
def _assistant_text(s):
    msgs = s.get('messages', [])
    for m in msgs:
        if isinstance(m, dict) and m.get('role') == 'assistant':
            return m.get('content') or ''
    return s.get('output') or ''

print('Corpus cleanliness gate (train split):')
gate_check(train, get_text=_assistant_text)
print('Corpus cleanliness gate (valid split):')
gate_check(valid, get_text=_assistant_text)


def save_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    print(f'  {path.name}: {len(data)} samples')


# Full format (with task_type / source metadata)
save_jsonl(train, DATA_OUT / 'train.jsonl')
save_jsonl(valid, DATA_OUT / 'valid.jsonl')   # NOTE: valid (not val) — mlx-lm convention
save_jsonl(test,  DATA_OUT / 'test.jsonl')

# mlx-lm format: only the "messages" key (mlx_lm.lora --use-chat-template reads this)
save_jsonl([{'messages': s['messages']} for s in train], MLX_DIR / 'train.jsonl')
save_jsonl([{'messages': s['messages']} for s in valid], MLX_DIR / 'valid.jsonl')

# Stats
stats = {
    'total': len(final), 'train': len(train), 'valid': len(valid), 'test': len(test),
    'task_counts': dict(type_counts),
    'avg_user_len':      int(sum(len(s['messages'][1]['content']) for s in final) / max(1, len(final))),
    'avg_assistant_len': int(sum(len(s['messages'][2]['content']) for s in final) / max(1, len(final))),
}
with open(DATA_OUT / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\nstats.json: avg user={stats["avg_user_len"]} chars, avg assistant={stats["avg_assistant_len"]} chars')


## Sanity check

In [ ]:
for s in random.sample(train, 3):
    print('─' * 60)
    print(f'Task: {s["task_type"]}')
    print(f'User:      {s["messages"][1]["content"][:120]}...')
    print(f'Assistant: {s["messages"][2]["content"][:120]}...')
print('─' * 60)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '16_dataset_assembly.png'

_task_counts = dict(type_counts)
_labels = list(_task_counts.keys())
_sizes  = list(_task_counts.values())
_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22'][:len(_labels)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: donut — task type composition ──────────────────────────────────────
wedges, _, autotexts = ax1.pie(
    _sizes, labels=_labels, colors=_colors, autopct='%1.0f%%',
    startangle=90, explode=[0.03] * len(_labels),
    pctdistance=0.78, wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for t in autotexts:
    t.set_fontsize(9); t.set_fontweight('bold')
ax1.add_artist(plt.Circle((0, 0), 0.50, fc='white'))
ax1.text(0, 0, f'{sum(_sizes):,}\nsamples', ha='center', va='center',
         fontsize=12, fontweight='bold', color='#2c3e50')
ax1.set_title('Task Type Composition', fontweight='bold', pad=15)

# ── Right: bar — train / valid / test split ───────────────────────────────────
_split_labels  = ['Train', 'Valid', 'Test']
_split_values  = [len(train), len(valid), len(test)]
_split_colors  = ['#3498db', '#f39c12', '#2ecc71']
bars = ax2.bar(_split_labels, _split_values, color=_split_colors, edgecolor='white', width=0.5)
ax2.set_ylabel('Samples')
ax2.set_title('Train / Valid / Test Split', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, n in zip(bars, _split_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, n + max(_split_values) * 0.01,
             f'{n:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

fig.suptitle(
    f'Dataset Assembly — {sum(_sizes):,} total  ·  '
    f'avg {stats["avg_user_len"]} chars user  /  {stats["avg_assistant_len"]} chars assistant',
    fontsize=13, fontweight='bold', y=1.01
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
# ── CSV export for downstream analysis ────────────────────────────────────────
import csv
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
csv_path = _run_dir / '16_dataset_assembly.csv'

def _has_code(text):
    return '```' in text

_all_labelled = (
    [(s, 'train') for s in train] +
    [(s, 'valid') for s in valid] +
    [(s, 'test')  for s in test]
)

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['task_type', 'source_notebook', 'split',
                     'user_length', 'assistant_length', 'has_code'])
    for s, split in _all_labelled:
        msgs = s.get('messages', [])
        user_text = msgs[1]['content'] if len(msgs) > 1 else ''
        asst_text = msgs[2]['content'] if len(msgs) > 2 else ''
        writer.writerow([
            s.get('task_type', ''),
            s.get('source', ''),
            split,
            len(user_text),
            len(asst_text),
            _has_code(asst_text),
        ])

print(f'Exported {len(_all_labelled)} rows to {csv_path}')

## Summary

In [ ]:
print('=' * 65)
print('notebook 12 — DATASET ASSEMBLY SUMMARY')
print('=' * 65)

_tc = dict(type_counts)
_total = sum(_tc.values())

print(f'\nInput sources:')
print(f'  Synthetic samples (NB10/13):  {len([s for s in samples if s.get("valid") is not False and "domain" in s]):>6,}')
print(f'  Curated pairs (NB04+05-09):   {kp_added:>6,}')

print(f'\nPipeline funnel:')
print(f'  Raw combined:       {len(samples):>6,}')
print(f'  After ChatML conv:  {len(chatml):>6,}  (dropped {len(samples)-len(chatml)} empty/invalid)')
print(f'  After dedup:        {len(deduped):>6,}')
print(f'  After balance cap:  {_total:>6,}  (type-weighted caps)')
print(f'  After plausibility: {len(final):>6,}  (rejected {_total - len(final)} implausible)')
print(f'  After quality filt: {len(train)+len(valid)+len(test):>6,}')

print(f'\nTask type distribution (after cap):')
for t, n in sorted(_tc.items(), key=lambda x: -x[1]):
    bar = '█' * int(30 * n / max(1, _total))
    pct = 100 * n / max(1, _total)
    flag = '  ← majority' if pct > 40 else ''
    print(f'  {t:25s} {n:5d}  {pct:5.1f}%  {bar}{flag}')

print(f'\nFinal split:')
print(f'  Train: {len(train):,}  ({100*len(train)/max(1,len(train)+len(valid)+len(test)):.0f}%)')
print(f'  Valid: {len(valid):,}  ({100*len(valid)/max(1,len(train)+len(valid)+len(test)):.0f}%)')
print(f'  Test:  {len(test):,}  ({100*len(test)/max(1,len(train)+len(valid)+len(test)):.0f}%)')

print(f'\nAvg lengths (train):')
_u_lens = [len(s['messages'][1]['content']) for s in train]
_a_lens = [len(s['messages'][2]['content']) for s in train]
print(f'  User prompt:      {sum(_u_lens)//max(1,len(_u_lens)):,} chars  '
      f'  (min {min(_u_lens)}, max {max(_u_lens)})')
print(f'  Assistant answer: {sum(_a_lens)//max(1,len(_a_lens)):,} chars  '
      f'  (min {min(_a_lens)}, max {max(_a_lens)})')

print(f'\nOutput files:')
print(f'  {DATA_OUT / "train.jsonl"}')
print(f'  {DATA_OUT / "valid.jsonl"}')
print(f'  {MLX_DIR  / "train.jsonl"}  ← used by NB12')
print(f'  {DATA_OUT / "stats.json"}')
print('=' * 65)

## Data Quality Checks

In [ ]:
# ── Quality filters applied to training set ───────────────────────────────────
# Remove samples that are likely to hurt training:
#   1. Too short (< 30 chars user or < 20 chars assistant) — no signal
#   2. Too repetitive — assistant is just a copy of the user prompt (ratio > 80%)
#   3. Obvious junk — assistant is all punctuation / whitespace / numbers only
#   4. Placeholder outputs — literally "..." or "None" or empty code blocks

import re as _re

_MIN_USER_LEN   = 30
_MIN_ASST_LEN   = 20
_MAX_COPY_RATIO = 0.80   # flag if assistant overlaps > 80% of user tokens

def _token_set(text):
    return set(_re.findall(r'\w+', text.lower()))

def _is_junk_output(text):
    stripped = text.strip()
    if stripped in ('...', 'None', 'null', ''):
        return True
    # Pure punctuation / whitespace
    if _re.fullmatch(r'[^a-zA-Z0-9\n]+', stripped):
        return True
    # Exclamation-mark collapse (the exact symptom we observed post-training)
    if _re.fullmatch(r'!+', stripped):
        return True
    # Empty code fence
    if stripped in ('```aro\n```', '```\n```', '```aro```'):
        return True
    return False

def _is_copy(user, asst):
    u_tok = _token_set(user)
    a_tok = _token_set(asst)
    if not a_tok:
        return True
    overlap = len(u_tok & a_tok) / len(a_tok)
    return overlap > _MAX_COPY_RATIO

quality_issues = []
clean_train = []
for s in train:
    user = s['messages'][1]['content']
    asst = s['messages'][2]['content']

    if len(user) < _MIN_USER_LEN:
        quality_issues.append(('too_short_user',  user[:80]))
        continue
    if len(asst) < _MIN_ASST_LEN:
        quality_issues.append(('too_short_asst',  asst[:80]))
        continue
    if _is_junk_output(asst):
        quality_issues.append(('junk_output',     asst[:80]))
        continue
    if _is_copy(user, asst):
        quality_issues.append(('copy_of_prompt',  asst[:80]))
        continue
    clean_train.append(s)

removed = len(train) - len(clean_train)
print(f'Quality filter: removed {removed} samples from train set')
print(f'  Remaining train: {len(clean_train)}')
if quality_issues:
    issue_counts = Counter(k for k, _ in quality_issues)
    for issue, cnt in issue_counts.most_common():
        print(f'  {issue}: {cnt}')
    if removed > 0:
        print('\nExample removed samples:')
        for issue, snippet in quality_issues[:3]:
            print(f'  [{issue}] {repr(snippet[:60])}')

# Guard: fail if more than 20% of the training set was junk
if removed / max(1, len(train)) > 0.20:
    raise ValueError(
        f'{removed}/{len(train)} train samples ({removed/len(train):.0%}) failed quality checks. '
        f'Investigate the data pipeline — too much junk to proceed safely.'
    )

# Replace train with the clean version
train = clean_train
print(f'\n✓ Quality check passed — {len(train)} clean training samples')

# Rewrite mlx train.jsonl with clean samples
def save_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

save_jsonl([{'messages': s['messages']} for s in train], MLX_DIR / 'train.jsonl')
print(f'Rewrote: {MLX_DIR / "train.jsonl"}')